In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 【读取步骤】
# 读取最终清洗后的训练集和测试集
# ============================================================

model_train = pd.read_parquet(
    "data/processed/model_train_clean.parquet"
)

model_test = pd.read_parquet(
    "data/processed/model_test_clean.parquet"
)


# ============================================================
# 【准备步骤】
# 分离客户编号、特征和目标变量
# ============================================================

# 1. 保存测试集客户编号
test_customer_ids = (
    model_test["SK_ID_CURR"]
    .copy()
)


# 2. 提取目标变量
y = (
    model_train["TARGET"]
    .astype("int8")
    .copy()
)


# 3. 建立训练特征
X = (
    model_train
    .drop(
        columns=[
            "TARGET",
            "SK_ID_CURR"
        ]
    )
    .copy()
)


# 4. 建立测试特征
X_test = (
    model_test
    .drop(
        columns=[
            "SK_ID_CURR"
        ]
    )
    .copy()
)


# 5. 划分训练集和验证集
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("X_train Shape:", X_train.shape)
print("X_valid Shape:", X_valid.shape)
print("X_test Shape:", X_test.shape)

print("训练集违约率:", y_train.mean())
print("验证集违约率:", y_valid.mean())

X_train Shape: (246008, 439)
X_valid Shape: (61503, 439)
X_test Shape: (48744, 439)
训练集违约率: 0.08072908198107379
验证集违约率: 0.08072776937710356


In [2]:
# ============================================================
# 【检查步骤】
# 区分类别特征和数值特征
# ============================================================

categorical_columns = (
    X_train
    .select_dtypes(
        include=[
            "object",
            "category"
        ]
    )
    .columns
    .tolist()
)


numeric_columns = (
    X_train
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)


print(
    "类别特征数量:",
    len(categorical_columns)
)

print(
    "数值特征数量:",
    len(numeric_columns)
)

print(
    "总特征数量:",
    len(categorical_columns)
    + len(numeric_columns)
)

print(
    "X_train 字段数量:",
    X_train.shape[1]
)

类别特征数量: 24
数值特征数量: 415
总特征数量: 439
X_train 字段数量: 439


In [3]:
# ============================================================
# 【数据预处理】
# 类别特征独热编码，数值特征标准化
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_columns
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_columns
        )
    ],
    remainder="drop"
)

In [4]:
# ============================================================
# 【基线模型】
# 建立 Logistic Regression Pipeline
# ============================================================

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                solver="saga",
                class_weight="balanced",
                max_iter=300,
                tol=1e-3,
                random_state=42
            )
        )
    ]
)

In [5]:
# ============================================================
# 【模型训练】
# 训练 Logistic Regression
# ============================================================

logistic_pipeline.fit(
    X_train,
    y_train
)

print(
    "Logistic Regression 训练完成"
)

Logistic Regression 训练完成


/Users/hulk/Documents/面试项目/credit-risk-portfolio-analytics/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [6]:
# ============================================================
# 【模型预测】
# 生成验证集违约概率和类别预测
# ============================================================

# 1. 预测违约概率
valid_probabilities = (
    logistic_pipeline
    .predict_proba(X_valid)[:, 1]
)


# 2. 使用0.5作为暂时分类阈值
valid_predictions = (
    valid_probabilities >= 0.5
).astype("int8")


print(
    "预测概率最小值:",
    valid_probabilities.min()
)

print(
    "预测概率最大值:",
    valid_probabilities.max()
)

print(
    "预测为违约的客户比例:",
    valid_predictions.mean()
)

预测概率最小值: 2.0757265004388985e-06
预测概率最大值: 0.9982839384070398
预测为违约的客户比例: 0.3200331691137018


In [7]:
# ============================================================
# 【模型评估】
# 计算 ROC-AUC、PR-AUC 和分类指标
# ============================================================

roc_auc = roc_auc_score(
    y_valid,
    valid_probabilities
)

pr_auc = average_precision_score(
    y_valid,
    valid_probabilities
)


print(
    "Validation ROC-AUC:",
    round(roc_auc, 4)
)

print(
    "Validation PR-AUC:",
    round(pr_auc, 4)
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_valid,
        valid_predictions,
        digits=4
    )
)


print(
    "Confusion Matrix:"
)

print(
    confusion_matrix(
        y_valid,
        valid_predictions
    )
)

Validation ROC-AUC: 0.7803
Validation PR-AUC: 0.2701

Classification Report:
              precision    recall  f1-score   support

           0     0.9654    0.7141    0.8209     56538
           1     0.1787    0.7086    0.2855      4965

    accuracy                         0.7136     61503
   macro avg     0.5721    0.7113    0.5532     61503
weighted avg     0.9019    0.7136    0.7777     61503

Confusion Matrix:
[[40373 16165]
 [ 1447  3518]]


In [8]:
# ============================================================
# 【阈值分析】
# 比较不同阈值下 Precision、Recall 和 F1
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)


# 1. 设置需要比较的分类阈值
thresholds = np.arange(
    0.10,
    0.81,
    0.05
)


# 2. 保存每个阈值的模型表现
threshold_results = []

for threshold in thresholds:

    threshold_predictions = (
        valid_probabilities >= threshold
    ).astype("int8")

    threshold_results.append(
        {
            "threshold": threshold,
            "precision": precision_score(
                y_valid,
                threshold_predictions,
                zero_division=0
            ),
            "recall": recall_score(
                y_valid,
                threshold_predictions,
                zero_division=0
            ),
            "f1_score": f1_score(
                y_valid,
                threshold_predictions,
                zero_division=0
            ),
            "accuracy": accuracy_score(
                y_valid,
                threshold_predictions
            ),
            "predicted_default_rate": (
                threshold_predictions.mean()
            )
        }
    )


# 3. 转换成DataFrame
threshold_comparison = pd.DataFrame(
    threshold_results
)


# 4. 查看结果
threshold_comparison.round(4)

,threshold,precision,recall,f1_score,accuracy,predicted_default_rate
0,0.10,0.0852,0.9927,0.1570,0.1391,0.9405
1,0.15,0.0914,0.9809,0.1672,0.2114,0.8663
2,0.20,0.0994,0.9621,0.1802,0.2933,0.7813
3,0.25,0.1091,0.9360,0.1954,0.3777,0.6927
4,0.30,0.1196,0.9011,0.2112,0.4567,0.6081
5,0.35,0.1320,0.8634,0.2289,0.5305,0.5282
6,0.40,0.1459,0.8163,0.2476,0.5994,0.4517
7,0.45,0.1613,0.7644,0.2663,0.6600,0.3826
8,0.50,0.1787,0.7086,0.2855,0.7136,0.3200
9,0.55,0.1989,0.6451,0.3040,0.7616,0.2619


In [9]:
# ============================================================
# 【检查步骤】
# 找出 F1 Score 最高的分类阈值
# ============================================================

best_f1_result = (
    threshold_comparison
    .sort_values(
        "f1_score",
        ascending=False
    )
    .head(1)
)

best_f1_result

,threshold,precision,recall,f1_score,accuracy,predicted_default_rate
12,0.7,0.281076,0.416717,0.335713,0.866868,0.119685


In [10]:
# ============================================================
# 【业务阈值检查】
# 查看 Recall 至少达到70%的阈值
# ============================================================

threshold_comparison[
    threshold_comparison["recall"] >= 0.70
].sort_values(
    "precision",
    ascending=False
).round(4)

,threshold,precision,recall,f1_score,accuracy,predicted_default_rate
8,0.50,0.1787,0.7086,0.2855,0.7136,0.3200
7,0.45,0.1613,0.7644,0.2663,0.6600,0.3826
6,0.40,0.1459,0.8163,0.2476,0.5994,0.4517
5,0.35,0.1320,0.8634,0.2289,0.5305,0.5282
4,0.30,0.1196,0.9011,0.2112,0.4567,0.6081
3,0.25,0.1091,0.9360,0.1954,0.3777,0.6927
2,0.20,0.0994,0.9621,0.1802,0.2933,0.7813
1,0.15,0.0914,0.9809,0.1672,0.2114,0.8663
0,0.10,0.0852,0.9927,0.1570,0.1391,0.9405


In [11]:
# ============================================================
# 【阈值结果】
# 比较业务召回阈值与最高F1阈值
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)


# 1. 设置两个候选阈值
selected_thresholds = {
    "Recall-focused threshold": 0.50,
    "Best F1 threshold": 0.70
}


# 2. 保存评估结果
selected_threshold_results = []

for threshold_name, threshold in selected_thresholds.items():

    predictions = (
        valid_probabilities >= threshold
    ).astype("int8")

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        predictions
    ).ravel()

    selected_threshold_results.append(
        {
            "threshold_name": threshold_name,
            "threshold": threshold,
            "precision": precision_score(
                y_valid,
                predictions
            ),
            "recall": recall_score(
                y_valid,
                predictions
            ),
            "f1_score": f1_score(
                y_valid,
                predictions
            ),
            "accuracy": accuracy_score(
                y_valid,
                predictions
            ),
            "predicted_default_rate": predictions.mean(),
            "true_negative": tn,
            "false_positive": fp,
            "false_negative": fn,
            "true_positive": tp
        }
    )


selected_threshold_summary = pd.DataFrame(
    selected_threshold_results
)

selected_threshold_summary.round(4)

,threshold_name,threshold,precision,recall,f1_score,accuracy,predicted_default_rate,true_negative,false_positive,false_negative,true_positive
0,Recall-focused threshold,0.5,0.1787,0.7086,0.2855,0.7136,0.3200,40373,16165,1447,3518
1,Best F1 threshold,0.7,0.2811,0.4167,0.3357,0.8669,0.1197,51246,5292,2896,2069


In [12]:
# ============================================================
# 【保存结果】
# 保存阈值比较结果
# ============================================================

selected_threshold_summary.to_csv(
    "data/processed/logistic_threshold_comparison.csv",
    index=False
)

threshold_comparison.to_csv(
    "data/processed/logistic_all_threshold_results.csv",
    index=False
)

print("阈值结果保存完成")

阈值结果保存完成
